> **Superseded.** This notebook is an earlier, unsuccessful attempt to decode SCiO scans locally. The scans are encrypted on the device; see `08_scio_keyrecovery.ipynb` and the README for the current approach. Kept for history.

# SCiO Offline Raw-to-331 Spectrum Experiments

This notebook is separate from `01_scio_usb.ipynb` on purpose. The USB notebook captures raw data; this notebook tests hypotheses for converting one complete raw scan package into one 331-band spectrum.

Each supervised example uses the full package:

- `sample`
- `sample_dark`
- `sample_white`
- `sample_white_dark`
- optional `sample_gradient`
- optional `sample_white_gradient`

The notebook must derive candidate spectra from the raw `sample*` fields. The saved `spec_data` is used only as a reference target for scoring and visual comparison. The old big-endian 400-integer unpacking is included as a rejected baseline because matching byte length is not evidence that it is the physical spectrum.

In [ ]:
from pathlib import Path
import base64
import csv
import json
import math
import struct

import numpy as np
import matplotlib.pyplot as plt

DATA_DIR = Path('..') / 'data_scans'
SCAN_KEYS = [
    'sample',
    'sample_dark',
    'sample_white',
    'sample_white_dark',
    'sample_gradient',
    'sample_white_gradient',
]
TARGET_LEN = 331
WAVELENGTHS_NM = np.arange(740, 740 + TARGET_LEN)
TOP_N_OUTPUTS = 5


In [ ]:
def _decode_blob(value, *, is_hex=False):
    if value is None:
        return None
    if isinstance(value, bytes):
        return value
    text = ''.join(str(value).split())
    if is_hex:
        return bytes.fromhex(text)
    # Captures can contain either standard Base64 from Android logs or URL-safe Base64
    # from the USB notebook. urlsafe_b64decode accepts both once padding is restored.
    text += '=' * ((4 - len(text) % 4) % 4)
    return base64.urlsafe_b64decode(text)


def _parse_spec_data(value):
    if isinstance(value, str):
        return np.asarray(json.loads(value), dtype=float)
    return np.asarray(value, dtype=float)


def load_scan_package(path):
    path = Path(path)
    data = json.loads(path.read_text())
    b64_data = data.get('b64_data', {})
    raw_data = data.get('raw_data', {})
    blobs = {}
    for key in SCAN_KEYS:
        if key in b64_data:
            blobs[key] = _decode_blob(b64_data[key])
        elif key in raw_data:
            blobs[key] = _decode_blob(raw_data[key], is_hex=True)
    target = _parse_spec_data(data['spec_data']) if 'spec_data' in data else None
    return {'path': path, 'meta': data.get('device', {}), 'blobs': blobs, 'target': target}


def load_all_examples(data_dir=DATA_DIR):
    examples = [load_scan_package(p) for p in sorted(Path(data_dir).glob('*.json'))]
    for ex in examples:
        if ex['target'] is not None and len(ex['target']) != TARGET_LEN:
            raise ValueError(f"{ex['path']} target has {len(ex['target'])} points, expected {TARGET_LEN}")
    return examples

examples = load_all_examples()
print(f'Loaded {len(examples)} scan packages')
for ex in examples:
    print(ex['path'].name, 'target_len=', None if ex['target'] is None else len(ex['target']))
    for key, blob in ex['blobs'].items():
        u32 = struct.unpack('<I', blob[:4])[0] if len(blob) >= 4 else None
        print(f'  {key:22s} bytes={len(blob):4d} u32le_header={u32}')

## Candidate Numeric And CMOS/Image-Like Views

The raw scan fields are not treated as final spectra. Each candidate below starts from the raw `sample*` blobs and tries a specific interpretation:

- numeric endian/offset views of the bytes;
- small CMOS-like image crops such as `20x20`, `25x16`, `40x10`, and related layouts;
- projections through rows, columns, diagonals, stripes, curved traces, and local windows;
- axis flips, transposes, memory order variants, and reversed spectral direction.

These are hypothesis tests. `spec_data` is only used later for scoring.


In [ ]:
VIEW_OFFSETS = [0, 4, 8, 16, 52, 110, 196]
VIEW_DTYPES = [
    ('u8', np.uint8),
    ('i8', np.int8),
    ('u16le', '<u2'),
    ('u16be', '>u2'),
    ('i16le', '<i2'),
    ('i16be', '>i2'),
    ('u32le', '<u4'),
    ('u32be', '>u4'),
    ('i32le', '<i4'),
    ('i32be', '>i4'),
    ('f32le', '<f4'),
    ('f32be', '>f4'),
]
PREFERRED_VIEWS = {
    'u8_skip4', 'i8_skip4',
    'u16le_skip4', 'u16be_skip4', 'i16le_skip4', 'i16be_skip4',
    'u32le_skip4', 'u32be_skip4', 'i32le_skip4', 'i32be_skip4',
    'f32le_skip4', 'f32be_skip4',
    'u32be_400_rejected',
}
CMOS_IMAGE_VIEWS = {
    'u8_skip4', 'u16le_skip4', 'u16be_skip4', 'i16le_skip4', 'i16be_skip4',
    'u32le_skip4', 'u32be_skip4', 'f32le_skip4', 'f32be_skip4',
    'u32be_400_rejected',
}
BASE_IMAGE_SHAPES = [
    (20, 20), (25, 16), (16, 25), (40, 10), (10, 40), (50, 8), (8, 50),
    (23, 18), (18, 23), (30, 15), (15, 30),
]
IMAGE_METHODS = [
    'flat', 'row_sum', 'col_sum', 'row_mean', 'col_mean',
    'main_diag', 'anti_diag', 'diag_band', 'anti_diag_band',
    'center_row', 'center_col', 'stripe_h3', 'stripe_v3',
]
IMAGE_ORIENTATIONS = ['none', 'fliplr', 'transpose']
IMAGE_ORDERS = ['C', 'F']
CORRECTION_METHODS = [
    'dark_white_ratio', 'ratio_only', 'dark_subtract', 'white_subtract',
    'white_ratio', 'log_dark_white', 'clipped_dark_white',
]
POST_TRANSFORMS = ['raw', 'robust', 'smooth5', 'derivative', 'log_abs']
GRADIENT_MODES = ['none', 'ratio', 'subtract']


def _offset_label(offset):
    if offset == 0:
        return 'all'
    return f'skip{offset}'


def _array_from_offset(blob, dtype, offset):
    itemsize = np.dtype(dtype).itemsize
    usable = len(blob) - offset
    if usable < itemsize:
        return None
    usable -= usable % itemsize
    if usable <= 0:
        return None
    arr = np.frombuffer(blob[offset:offset + usable], dtype=dtype).astype(float)
    if arr.size == 0:
        return None
    return arr


def numeric_views(blob):
    """Return endian/offset interpretations of one raw blob."""
    views = {}
    if blob is None:
        return views
    for offset in VIEW_OFFSETS:
        if offset >= len(blob):
            continue
        label = _offset_label(offset)
        for name, dtype in VIEW_DTYPES:
            arr = _array_from_offset(blob, dtype, offset)
            if arr is None:
                continue
            views[f'{name}_{label}'] = arr

    # Rejected baseline kept for comparison with an earlier incorrect idea:
    # four header bytes, then 400 big-endian uint32 values, then ignored tail.
    if len(blob) >= 4 + 400 * 4:
        views['u32be_400_rejected'] = np.asarray(struct.unpack('>4x400I', blob[:4 + 400 * 4]), dtype=float)
    return views


def candidate_shapes(n):
    """Return bounded CMOS-like image shapes that fit inside a 1D view."""
    shapes = []
    for shape in BASE_IMAGE_SHAPES:
        if shape[0] * shape[1] <= n and shape not in shapes:
            shapes.append(shape)

    # Add a few compact factor shapes when the array length suggests them.
    for rows in range(6, 61):
        if n % rows == 0:
            cols = n // rows
            if 6 <= cols <= 80 and rows * cols <= n:
                shape = (rows, cols)
                if shape not in shapes:
                    shapes.append(shape)
        if len(shapes) >= 16:
            break
    return shapes


ex0 = examples[0]
for key in ['sample', 'sample_dark', 'sample_white', 'sample_white_dark', 'sample_gradient']:
    if key in ex0['blobs']:
        views = numeric_views(ex0['blobs'][key])
        preview = {name: len(views[name]) for name in sorted(views) if name in PREFERRED_VIEWS}
        print(key, preview)


## Package-Level Projection And Correction

A candidate decoder must consume the complete raw scan package and return exactly one 331-value spectrum. The correction methods below intentionally test several possible orders because the Android app does not contain the proprietary offline transform.


In [ ]:
def _finite_clean(y):
    y = np.asarray(y, dtype=float)
    if y.size == 0:
        return y
    finite = np.isfinite(y)
    if not finite.any():
        return np.zeros_like(y, dtype=float)
    fill = np.nanmedian(y[finite])
    return np.where(finite, y, fill)


def resample_to_len(y, n):
    y = _finite_clean(y)
    if y.size == 0:
        return np.full(n, np.nan)
    if y.size == 1:
        return np.full(n, float(y[0]))
    x_old = np.linspace(0, 1, y.size)
    x_new = np.linspace(0, 1, n)
    return np.interp(x_new, x_old, y)


def resample_to_331(y):
    return resample_to_len(y, TARGET_LEN)


def safe_ratio(num, den, eps=1e-9):
    num = _finite_clean(num)
    den = _finite_clean(den)
    n = max(num.size, den.size)
    if n == 0:
        return np.array([], dtype=float)
    num = resample_to_len(num, n)
    den = resample_to_len(den, n)
    scale = np.nanmedian(np.abs(den))
    guard = eps if not np.isfinite(scale) or scale == 0 else eps * max(1.0, scale)
    return num / np.where(np.abs(den) < guard, np.nan, den)


def as_image(arr, shape, order='C'):
    arr = _finite_clean(arr)
    rows, cols = shape
    needed = rows * cols
    if needed > arr.size:
        return None
    return np.asarray(arr[:needed], dtype=float).reshape(shape, order=order)


def orient_image(img, orientation):
    if orientation == 'none':
        return img
    if orientation == 'fliplr':
        return np.fliplr(img)
    if orientation == 'flipud':
        return np.flipud(img)
    if orientation == 'transpose':
        return img.T
    if orientation == 'rot90':
        return np.rot90(img, 1)
    if orientation == 'rot270':
        return np.rot90(img, 3)
    raise ValueError(f'unknown orientation: {orientation}')


def _diag_band(img, anti=False):
    src = np.fliplr(img) if anti else img
    bands = [np.diag(src, k=k) for k in (-1, 0, 1)]
    max_len = max((b.size for b in bands), default=0)
    if max_len == 0:
        return np.array([], dtype=float)
    return np.nanmean([resample_to_len(b, max_len) for b in bands if b.size], axis=0)


def _curved_diag(img, power):
    rows, cols = img.shape
    if cols <= 1:
        return img[:, 0]
    xs = np.linspace(0, 1, cols)
    ys = np.rint((rows - 1) * np.power(xs, power)).astype(int)
    return img[ys, np.arange(cols)]


def _local_windows(img, windows=12):
    rows, cols = img.shape
    values = []
    for x0 in np.linspace(0, max(cols - 1, 0), windows):
        c = int(round(x0))
        r = int(round((rows - 1) * c / max(cols - 1, 1)))
        r0, r1 = max(0, r - 1), min(rows, r + 2)
        c0, c1 = max(0, c - 1), min(cols, c + 2)
        values.append(np.nanmean(img[r0:r1, c0:c1]))
    return np.asarray(values, dtype=float)


def project_array(arr, method, shape=None, order='C', orientation='none'):
    arr = _finite_clean(arr)
    if method == 'identity' or shape is None:
        return arr
    img = as_image(arr, shape, order=order)
    if img is None:
        return None
    img = orient_image(img, orientation)
    if method == 'flat':
        return img.ravel(order=order)
    if method == 'row_sum':
        return np.nansum(img, axis=1)
    if method == 'col_sum':
        return np.nansum(img, axis=0)
    if method == 'row_mean':
        return np.nanmean(img, axis=1)
    if method == 'col_mean':
        return np.nanmean(img, axis=0)
    if method == 'row_max':
        return np.nanmax(img, axis=1)
    if method == 'col_max':
        return np.nanmax(img, axis=0)
    if method == 'main_diag':
        return np.diag(img)
    if method == 'anti_diag':
        return np.diag(np.fliplr(img))
    if method == 'diag_band':
        return _diag_band(img, anti=False)
    if method == 'anti_diag_band':
        return _diag_band(img, anti=True)
    if method == 'center_row':
        return img[img.shape[0] // 2, :]
    if method == 'center_col':
        return img[:, img.shape[1] // 2]
    if method == 'stripe_h3':
        r = img.shape[0] // 2
        return np.nanmean(img[max(0, r - 1):min(img.shape[0], r + 2), :], axis=0)
    if method == 'stripe_v3':
        c = img.shape[1] // 2
        return np.nanmean(img[:, max(0, c - 1):min(img.shape[1], c + 2)], axis=1)
    if method == 'curved_diag_low':
        return _curved_diag(img, 0.65)
    if method == 'curved_diag_high':
        return _curved_diag(img, 1.55)
    if method == 'local_windows':
        return _local_windows(img)
    raise ValueError(f'unknown projection method: {method}')


def post_transform(y, name):
    y = _finite_clean(y)
    if name == 'raw':
        return y
    if name == 'robust':
        med = np.nanmedian(y)
        q75, q25 = np.nanpercentile(y, [75, 25])
        scale = q75 - q25
        if not np.isfinite(scale) or scale == 0:
            scale = np.nanstd(y) or 1.0
        return (y - med) / scale
    if name == 'smooth5':
        if y.size < 5:
            return y
        kernel = np.ones(5, dtype=float) / 5.0
        return np.convolve(y, kernel, mode='same')
    if name == 'derivative':
        if y.size < 2:
            return y
        return np.gradient(y)
    if name == 'log_abs':
        return np.sign(y) * np.log1p(np.abs(y))
    raise ValueError(f'unknown post transform: {name}')


def apply_correction(projected, correction):
    s = projected['sample']
    d = projected['sample_dark']
    w = projected['sample_white']
    wd = projected['sample_white_dark']
    n = max(len(s), len(d), len(w), len(wd))
    s, d, w, wd = [resample_to_len(v, n) for v in (s, d, w, wd)]
    if correction == 'dark_white_ratio':
        return safe_ratio(s - d, w - wd)
    if correction == 'ratio_only':
        return safe_ratio(s, w)
    if correction == 'dark_subtract':
        return s - d
    if correction == 'white_subtract':
        return w - wd
    if correction == 'white_ratio':
        return safe_ratio(w, wd)
    if correction == 'log_dark_white':
        return np.log1p(np.abs(s - d)) - np.log1p(np.abs(w - wd))
    if correction == 'clipped_dark_white':
        ratio = safe_ratio(s - d, w - wd)
        lo, hi = np.nanpercentile(ratio, [1, 99])
        return np.clip(ratio, lo, hi)
    raise ValueError(f'unknown correction method: {correction}')


def decode_candidate(package, spec):
    blobs = package['blobs']
    required = ['sample', 'sample_dark', 'sample_white', 'sample_white_dark']
    if any(k not in blobs for k in required):
        return None
    views = {k: numeric_views(blobs[k]) for k in required}
    view_name = spec['view']
    if any(view_name not in views[k] for k in required):
        return None
    projected = {}
    for key in required:
        projected[key] = project_array(
            views[key][view_name],
            spec['method'],
            spec.get('shape'),
            spec.get('order', 'C'),
            spec.get('orientation', 'none'),
        )
        if projected[key] is None:
            return None
    corrected = apply_correction(projected, spec['correction'])

    gradient_mode = spec.get('gradient_mode', 'none')
    if gradient_mode != 'none' and 'sample_gradient' in blobs and 'sample_white_gradient' in blobs:
        gv = numeric_views(blobs['sample_gradient'])
        wgv = numeric_views(blobs['sample_white_gradient'])
        if view_name in gv and view_name in wgv:
            g_proj = project_array(gv[view_name], spec['method'], spec.get('shape'), spec.get('order', 'C'), spec.get('orientation', 'none'))
            wg_proj = project_array(wgv[view_name], spec['method'], spec.get('shape'), spec.get('order', 'C'), spec.get('orientation', 'none'))
            if g_proj is not None and wg_proj is not None:
                if gradient_mode == 'ratio':
                    corrected = safe_ratio(corrected, safe_ratio(g_proj, wg_proj))
                elif gradient_mode == 'subtract':
                    corrected = corrected - resample_to_len(g_proj - wg_proj, len(corrected))

    corrected = post_transform(corrected, spec.get('post', 'raw'))
    if spec.get('reverse', False):
        corrected = corrected[::-1]
    return resample_to_331(corrected)


def candidate_label(spec):
    shape = spec.get('shape')
    shape_text = 'none' if shape is None else f'{shape[0]}x{shape[1]}'
    return '|'.join([
        spec['view'], spec['method'], shape_text, spec.get('order', 'C'),
        spec.get('orientation', 'none'), spec['correction'], spec.get('post', 'raw'),
        spec.get('gradient_mode', 'none'), 'rev' if spec.get('reverse') else 'fwd',
    ])


## Scoring Candidate Decoders

Scores are diagnostic. With only a few paired examples, low error can still be overfit or accidental. The constant and shuffled baselines make it easier to reject candidates that are not better than trivial spectra.


In [ ]:
def _pearson(a, b):
    a = _finite_clean(a)
    b = _finite_clean(b)
    if a.size != b.size or a.size < 2:
        return np.nan
    if np.nanstd(a) == 0 or np.nanstd(b) == 0:
        return np.nan
    return float(np.corrcoef(a, b)[0, 1])


def _spectral_angle(a, b):
    a = _finite_clean(a)
    b = _finite_clean(b)
    denom = np.linalg.norm(a) * np.linalg.norm(b)
    if denom == 0 or not np.isfinite(denom):
        return np.nan
    cosang = np.clip(float(np.dot(a, b) / denom), -1.0, 1.0)
    return float(np.arccos(cosang))


def score_prediction(pred, target):
    pred = np.asarray(pred, dtype=float)
    target = np.asarray(target, dtype=float)
    if pred.size != TARGET_LEN or target.size != TARGET_LEN:
        return {'rmse': np.inf, 'mae': np.inf, 'corr': np.nan, 'derivative_corr': np.nan, 'spectral_angle': np.nan, 'norm_rmse': np.inf}
    pred = _finite_clean(pred)
    target = _finite_clean(target)
    err = pred - target
    rmse = float(np.sqrt(np.nanmean(err ** 2)))
    mae = float(np.nanmean(np.abs(err)))
    target_range = np.nanmax(target) - np.nanmin(target)
    norm_rmse = float(rmse / target_range) if np.isfinite(target_range) and target_range else np.inf
    return {
        'rmse': rmse,
        'mae': mae,
        'corr': _pearson(pred, target),
        'derivative_corr': _pearson(np.gradient(pred), np.gradient(target)),
        'spectral_angle': _spectral_angle(pred, target),
        'norm_rmse': norm_rmse,
    }


def baseline_rows_for_example(ex):
    target = ex['target']
    if target is None:
        return []
    constant = np.full(TARGET_LEN, np.nanmean(target))
    shuffled = target.copy()
    rng = np.random.default_rng(12345)
    rng.shuffle(shuffled)
    rows = []
    for name, pred in [('constant_mean_baseline', constant), ('shuffled_target_baseline', shuffled)]:
        score = score_prediction(pred, target)
        rows.append({
            'scan': ex['path'].name,
            'candidate': name,
            'view': 'baseline', 'method': name, 'shape': None, 'order': None,
            'orientation': None, 'correction': None, 'post': None,
            'gradient_mode': None, 'reverse': False,
            'prediction': pred,
            **score,
        })
    return rows


def iter_candidate_specs(package):
    sample_views = numeric_views(package['blobs'].get('sample'))
    view_names = [name for name in sorted(sample_views) if name in PREFERRED_VIEWS or name in CMOS_IMAGE_VIEWS]
    for view_name in view_names:
        for correction in CORRECTION_METHODS:
            for post in POST_TRANSFORMS:
                for gradient_mode in GRADIENT_MODES:
                    for reverse in [False, True]:
                        yield {
                            'view': view_name, 'method': 'identity', 'shape': None,
                            'order': 'C', 'orientation': 'none', 'correction': correction,
                            'post': post, 'gradient_mode': gradient_mode, 'reverse': reverse,
                        }

        if view_name not in CMOS_IMAGE_VIEWS:
            continue
        for shape in candidate_shapes(len(sample_views[view_name]))[:8]:
            for method in IMAGE_METHODS:
                orders = IMAGE_ORDERS if method in ('flat',) else ['C']
                for order in orders:
                    for orientation in IMAGE_ORIENTATIONS:
                        for correction in ['dark_white_ratio', 'ratio_only', 'dark_subtract', 'log_dark_white']:
                            for post in ['raw', 'robust']:
                                for gradient_mode in ['none', 'ratio']:
                                    for reverse in [False, True]:
                                        yield {
                                            'view': view_name, 'method': method, 'shape': shape,
                                            'order': order, 'orientation': orientation,
                                            'correction': correction, 'post': post,
                                            'gradient_mode': gradient_mode, 'reverse': reverse,
                                        }


def evaluate_candidates(examples, max_candidates_per_scan=None):
    rows = []
    skipped = []
    for ex in examples:
        target = ex['target']
        if target is None:
            continue
        rows.extend(baseline_rows_for_example(ex))
        for index, spec in enumerate(iter_candidate_specs(ex)):
            if max_candidates_per_scan is not None and index >= max_candidates_per_scan:
                break
            try:
                pred = decode_candidate(ex, spec)
            except Exception as exc:
                skipped.append({'scan': ex['path'].name, 'candidate': candidate_label(spec), 'reason': repr(exc)})
                continue
            if pred is None or pred.size != TARGET_LEN or not np.isfinite(pred).any():
                skipped.append({'scan': ex['path'].name, 'candidate': candidate_label(spec), 'reason': 'no finite 331-value prediction'})
                continue
            score = score_prediction(pred, target)
            rows.append({
                'scan': ex['path'].name,
                'candidate': candidate_label(spec),
                'view': spec['view'],
                'method': spec['method'],
                'shape': None if spec.get('shape') is None else f"{spec['shape'][0]}x{spec['shape'][1]}",
                'order': spec.get('order', 'C'),
                'orientation': spec.get('orientation', 'none'),
                'correction': spec['correction'],
                'post': spec.get('post', 'raw'),
                'gradient_mode': spec.get('gradient_mode', 'none'),
                'reverse': bool(spec.get('reverse', False)),
                'prediction': pred,
                **score,
            })
    rows.sort(key=lambda r: (not np.isfinite(r['rmse']), r['rmse'], -np.nan_to_num(r['corr'], nan=-999)))
    return rows, skipped


results, skipped_candidates = evaluate_candidates(examples, max_candidates_per_scan=10000)
print(f'Evaluated {len(results)} candidate rows; skipped {len(skipped_candidates)} invalid candidates.')
for ex in examples:
    scan_rows = [r for r in results if r['scan'] == ex['path'].name]
    best = next((r for r in scan_rows if not str(r['method']).endswith('baseline')), scan_rows[0] if scan_rows else None)
    constant = next((r for r in scan_rows if r['method'] == 'constant_mean_baseline'), None)
    if best:
        print('\n', ex['path'].name)
        print('  best:', {k: best[k] for k in ['rmse', 'mae', 'corr', 'derivative_corr', 'spectral_angle', 'view', 'method', 'shape', 'orientation', 'correction', 'post', 'gradient_mode', 'reverse']})
        if constant:
            print('  constant baseline rmse:', constant['rmse'])


In [ ]:
def best_candidate_rows_by_scan(results, rank=0, include_baselines=False):
    selected = {}
    for row in results:
        if not include_baselines and row['view'] == 'baseline':
            continue
        selected.setdefault(row['scan'], []).append(row)
    return {scan: rows[rank] if len(rows) > rank else None for scan, rows in selected.items()}


def plot_best_for_scan(scan_name=None, rank=0):
    if scan_name is None:
        if not results:
            raise ValueError('No candidate results were computed.')
        scan_name = results[0]['scan']
    selected = best_candidate_rows_by_scan(results, rank=rank)
    row = selected[scan_name]
    ex = next(e for e in examples if e['path'].name == scan_name)
    plt.figure(figsize=(10, 4))
    plt.plot(WAVELENGTHS_NM, ex['target'], label='reference spec_data', linewidth=2, color='black')
    plt.plot(WAVELENGTHS_NM, row['prediction'], label=f"rank {rank} derived", linewidth=1.2)
    plt.title(f"{scan_name}\n{row['candidate']}")
    plt.xlabel('Wavelength / inferred nm')
    plt.ylabel('Value')
    plt.legend()
    plt.tight_layout()
    return row


plot_best_for_scan(rank=0)


## Save Spectrum Images And Ranked Tables

This section writes PNG and CSV files to `notebooks/offline_decode_outputs/` so spectra can be inspected visually. Files whose names include `reference_spec_data` are the server/API target. Files whose names include `derived_candidate` are generated only from raw `sample*` fields.


In [ ]:
OUTPUT_DIR = Path('offline_decode_outputs')
OUTPUT_DIR.mkdir(exist_ok=True)


def _first_image_view(blob):
    """Return a stable image-like view for raw diagnostics."""
    if blob is None:
        return None
    views = numeric_views(blob)
    for name in ['u32be_400_rejected', 'u32le_skip4', 'u16le_skip4', 'u8_skip4']:
        arr = views.get(name)
        if arr is None:
            continue
        for shape in candidate_shapes(len(arr)):
            if shape == (20, 20):
                return as_image(arr, shape)
        shapes = candidate_shapes(len(arr))
        if shapes:
            return as_image(arr, shapes[0])
    return None


def save_raw_package_image_diagnostics(examples, output_dir=OUTPUT_DIR):
    saved = []
    for ex in examples:
        panels = []
        for key in ['sample', 'sample_dark', 'sample_gradient', 'sample_white', 'sample_white_dark', 'sample_white_gradient']:
            if key in ex['blobs']:
                img = _first_image_view(ex['blobs'][key])
                if img is not None:
                    panels.append((key, img))
        if not panels:
            continue
        cols = min(3, len(panels))
        rows = int(math.ceil(len(panels) / cols))
        fig, axes = plt.subplots(rows, cols, figsize=(4 * cols, 3.5 * rows), squeeze=False)
        for ax in axes.ravel():
            ax.axis('off')
        for ax, (key, img) in zip(axes.ravel(), panels):
            im = ax.imshow(img, cmap='viridis', aspect='auto')
            ax.set_title(key)
            ax.axis('on')
            fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
        fig.suptitle(f"Raw CMOS-like diagnostics: {ex['path'].name}")
        fig.tight_layout()
        out = output_dir / f"{ex['path'].stem}_raw_cmos_image_diagnostics.png"
        fig.savefig(out, dpi=160)
        plt.close(fig)
        saved.append(out)
    return saved


def save_target_spectrum_images(examples, output_dir=OUTPUT_DIR):
    """Save reference `spec_data` plots. These are not derived outputs."""
    saved = []
    for ex in examples:
        if ex['target'] is None:
            continue
        out = output_dir / f"{ex['path'].stem}_reference_spec_data.png"
        plt.figure(figsize=(10, 4))
        plt.plot(WAVELENGTHS_NM, ex['target'], color='black', linewidth=1.8)
        plt.xlabel('Wavelength / inferred nm')
        plt.ylabel('Reference spec_data value')
        plt.title(f"Reference server/API spec_data: {ex['path'].name}")
        plt.tight_layout()
        plt.savefig(out, dpi=160)
        plt.close()
        saved.append(out)
    return saved


def save_candidate_summary_csv(results, skipped, output_dir=OUTPUT_DIR):
    out = output_dir / 'candidate_summary.csv'
    fields = [
        'scan', 'candidate', 'view', 'method', 'shape', 'order', 'orientation',
        'correction', 'post', 'gradient_mode', 'reverse',
        'rmse', 'mae', 'corr', 'derivative_corr', 'spectral_angle', 'norm_rmse',
    ]
    with out.open('w', newline='', encoding='utf-8') as f:
        writer = csv.DictWriter(f, fieldnames=fields)
        writer.writeheader()
        for row in results:
            writer.writerow({field: row.get(field) for field in fields})

    skipped_out = output_dir / 'candidate_skipped.csv'
    with skipped_out.open('w', newline='', encoding='utf-8') as f:
        writer = csv.DictWriter(f, fieldnames=['scan', 'candidate', 'reason'])
        writer.writeheader()
        writer.writerows(skipped)
    return [out, skipped_out]


def save_derived_candidate_spectrum_images(examples, results, output_dir=OUTPUT_DIR, rank=0):
    """Save actual derived 331-point spectra from raw `sample*` fields."""
    saved = []
    selected = best_candidate_rows_by_scan(results, rank=rank)
    for ex in examples:
        row = selected.get(ex['path'].name)
        if row is None:
            continue
        out_png = output_dir / f"{ex['path'].stem}_derived_candidate_rank{rank}.png"
        out_csv = output_dir / f"{ex['path'].stem}_derived_candidate_rank{rank}.csv"
        pred = np.asarray(row['prediction'], dtype=float)
        np.savetxt(
            out_csv,
            np.column_stack([WAVELENGTHS_NM, pred]),
            delimiter=',',
            header='wavelength_nm,derived_value',
            comments='',
        )
        plt.figure(figsize=(10, 4))
        plt.plot(WAVELENGTHS_NM, pred, label='derived from raw sample fields', linewidth=1.4)
        if ex['target'] is not None:
            plt.plot(WAVELENGTHS_NM, ex['target'], label='reference spec_data', color='black', linestyle='--', alpha=0.65)
        plt.xlabel('Wavelength / inferred nm')
        plt.ylabel('Value')
        plt.title(f"Derived candidate rank {rank}: {ex['path'].name}\n{row['candidate']}")
        plt.legend()
        plt.tight_layout()
        plt.savefig(out_png, dpi=160)
        plt.close()
        saved.extend([out_png, out_csv])
    return saved


def save_best_candidate_overlay_images(examples, results, output_dir=OUTPUT_DIR, rank=0):
    """Save target-vs-derived overlays for scans that have spec_data."""
    saved = []
    selected = best_candidate_rows_by_scan(results, rank=rank)
    for ex in examples:
        if ex['target'] is None:
            continue
        row = selected.get(ex['path'].name)
        if row is None:
            continue
        out = output_dir / f"{ex['path'].stem}_target_vs_derived_rank{rank}_overlay.png"
        plt.figure(figsize=(10, 4))
        plt.plot(WAVELENGTHS_NM, ex['target'], label='reference spec_data', color='black', linewidth=1.8)
        plt.plot(WAVELENGTHS_NM, row['prediction'], label=f"derived rmse={row['rmse']:.4g}, corr={row['corr']:.3f}", linewidth=1.2)
        plt.xlabel('Wavelength / inferred nm')
        plt.ylabel('Value')
        plt.title(f"Target vs derived rank {rank}: {ex['path'].name}\n{row['candidate']}")
        plt.legend()
        plt.tight_layout()
        plt.savefig(out, dpi=160)
        plt.close()
        saved.append(out)
    return saved


def save_top_n_outputs(examples, results, skipped, output_dir=OUTPUT_DIR, top_n=TOP_N_OUTPUTS):
    saved = []
    saved.extend(save_candidate_summary_csv(results, skipped, output_dir))
    saved.extend(save_raw_package_image_diagnostics(examples, output_dir))
    saved.extend(save_target_spectrum_images(examples, output_dir))
    for rank in range(top_n):
        saved.extend(save_derived_candidate_spectrum_images(examples, results, output_dir, rank=rank))
        saved.extend(save_best_candidate_overlay_images(examples, results, output_dir, rank=rank))
    return saved


saved_outputs = save_top_n_outputs(examples, results, skipped_candidates)
print('Saved ranked tables and spectrum/image outputs:')
for path in saved_outputs:
    print(' ', path)


## Firmware Parameter Blob Recovery Notes

The decompiled Android app defines the firmware parameter file IDs as:

| Name | ID | Why it matters |
|---|---:|---|
| `deadPixelsIndices` | `100` | Pixels to ignore before binning/projection. |
| `centers` | `101` | Likely spectral trace/bin center locations. |
| `bins` | `102` | Likely image-to-spectrum bin map. |
| `nPixelsPerBin` | `103` | Likely normalization counts for each bin. |

Important evidence from the app:

- `READ_FILE_HEADER` is used for IDs `100-103` to read checksums/headers from the device, not full parameter bodies.
- `FILE_DOWNLOAD` is used during firmware upgrade to write parameter files to the device. It is not proven to read files back.
- `FirmwareUpgradeModel.getByteData()` Base64-decodes a server-provided file and strips the first four checksum bytes.
- The firmware upgrade response parser looked for a `new_version` object with keys matching the enum names above.
- The app cached these Base64 firmware file strings in Android `SharedPreferences` under the enum names.

Best recovery paths:

1. Extract old app `SharedPreferences` from a phone, emulator, Android backup, rooted-device image, or emulator snapshot that connected while the server was alive.
2. Search HTTP/proxy/log backups for the firmware-upgrade response containing `new_version: {"centers": "...", "bins": "...", ...}`.
3. Use `read_file_header(100-103)` from `01_scio_usb.ipynb` to get the checksums your physical device expects, then match those checksums against recovered blobs.
4. Only after exhausting those paths, cautiously probe read-only USB/BLE file commands for undocumented body reads. Keep write commands disabled unless explicitly enabled.


In [ ]:
FIRMWARE_PARAM_IDS = {
    'deadPixelsIndices': 100,
    'centers': 101,
    'bins': 102,
    'nPixelsPerBin': 103,
}
FIRMWARE_PARAM_DIR = Path('recovered_firmware_params')


def _maybe_base64_decode(raw):
    """Decode text Base64 firmware blobs; leave binary blobs untouched."""
    stripped = raw.strip()
    if not stripped:
        return raw
    try:
        text = stripped.decode('ascii')
    except UnicodeDecodeError:
        return raw
    compact = ''.join(text.split())
    if not compact:
        return raw
    try:
        padding = '=' * (-len(compact) % 4)
        decoded = base64.urlsafe_b64decode(compact + padding)
    except Exception:
        return raw
    return decoded if decoded else raw


def load_firmware_blob(path, *, strip_checksum=True):
    """Load one recovered firmware parameter file.

    Server-style files include four checksum bytes at the beginning. That mirrors
    Android `FirmwareUpgradeModel.getByteData()`, which strips those bytes before
    sending data to the device.
    """
    raw_file = Path(path).read_bytes()
    raw = _maybe_base64_decode(raw_file)
    if strip_checksum and len(raw) >= 4:
        checksum_le = struct.unpack('<I', raw[:4])[0]
        checksum_be = struct.unpack('>I', raw[:4])[0]
        data = raw[4:]
    else:
        checksum_le = None
        checksum_be = None
        data = raw
    return {
        'path': Path(path),
        'checksum_le': checksum_le,
        'checksum_be': checksum_be,
        'data': data,
        'raw': raw,
    }


def discover_recovered_firmware_params(param_dir=FIRMWARE_PARAM_DIR):
    """Find recovered files by enum name or numeric ID.

    Put candidate files in `notebooks/recovered_firmware_params/`, for example:
    `centers.bin`, `bins.b64`, `101.txt`, or `nPixelsPerBin.raw`.
    """
    param_dir = Path(param_dir)
    found = {}
    if not param_dir.exists():
        return found
    for name, file_id in FIRMWARE_PARAM_IDS.items():
        candidates = []
        for stem in [name, str(file_id)]:
            candidates.extend(param_dir.glob(stem + '.*'))
            exact = param_dir / stem
            if exact.exists():
                candidates.append(exact)
        if candidates:
            found[name] = load_firmware_blob(candidates[0])
    return found


def parse_numeric_param_views(data):
    """Experimental numeric views for recovered parameter blobs."""
    return {
        'u8': np.frombuffer(data, dtype=np.uint8).astype(float),
        'u16le': np.frombuffer(data[:len(data) - len(data) % 2], dtype='<u2').astype(float),
        'u16be': np.frombuffer(data[:len(data) - len(data) % 2], dtype='>u2').astype(float),
        'i16le': np.frombuffer(data[:len(data) - len(data) % 2], dtype='<i2').astype(float),
        'i16be': np.frombuffer(data[:len(data) - len(data) % 2], dtype='>i2').astype(float),
        'f32le': np.frombuffer(data[:len(data) - len(data) % 4], dtype='<f4').astype(float),
        'f32be': np.frombuffer(data[:len(data) - len(data) % 4], dtype='>f4').astype(float),
    }


def image_to_spectrum_with_params(package, params):
    """Placeholder for the likely real transform once parameter blobs exist."""
    raise NotImplementedError('Recovered firmware parameter formats are needed before this can be implemented.')


recovered_params = discover_recovered_firmware_params()
if recovered_params:
    print('Recovered firmware parameter candidates:')
    for name, blob in recovered_params.items():
        print(name, blob['path'], 'data bytes:', len(blob['data']), 'checksum_le:', blob['checksum_le'])
else:
    print(f'No recovered firmware params found in {FIRMWARE_PARAM_DIR}.')
